# 01 — Visualizar a bola morta: redução de dimensão e testes de hipótese

**Pergunta.** Dá para *ver* a bola morta no movimento coletivo dos jogadores? E quais hipóteses físicas e
biológicas sobre esse movimento os dados sustentam?

**Entradas** (nada aqui lê o GT privado):

| fonte | o que é |
|---|---|
| `viewer/data/features_tactical.*` | 75 features por frame de `viewer/features.py` |
| `viewer/data/features_coletivas_tactical.*` | 50 features novas de `analise/features_coletivas.py` (perseguição, ordem, MSD, contexto passado) |
| `evaluation/gt/*_public.csv` | GT público, 5 min: **só 4 trechos de bola morta** |
| Smart Tagging | 59 SET PIECES com tipo: rótulo **fraco** no jogo inteiro |

**Regra estatística deste notebook.** Frames vizinhos são quase cópias um do outro, então a amostra efetiva são os
*eventos* (4 trechos D no GT, 59 set pieces), não os frames. Um Mann-Whitney sobre 7500 frames dá p ≈ 0 para qualquer
coisa. Por isso todo teste aqui é feito de uma destas formas:

* **deslocamento circular** (`rotulos.circular_shift_test`): o vetor de rótulos é deslocado em bloco, o que preserva
  a autocorrelação e a duração dos trechos, e o AUC observado é comparado com essa nula;
* **por evento**: o Cliff's δ de cada trecho morto contra a bola viva nos 30 s em volta, contando em quantos eventos o sinal é o
  previsto, ou um Wilcoxon pareado sobre os 59 set pieces, que são eventos independentes.

Pré-requisito, uma vez só: `python viewer/features.py` e `python analise/features_coletivas.py`.

In [ ]:
%matplotlib inline
import os, sys, json, warnings
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "4")   # evita o aviso do joblib/wmic no Windows
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")

HERE = os.path.abspath("") if os.path.exists("rotulos.py") else os.path.join(os.path.abspath(""), "analise")
sys.path.insert(0, HERE)
import rotulos as R
from features_coletivas import load_all
from explorar_estados import PLAYER_FEATURES

X, meta = load_all()
I, N, FPS = meta["index"], meta["n_frames"], R.FPS
DESC = {f["name"]: f["description"] for f in meta["features"]}
f = lambda name: X[:, I[name]]

gt = R.gt_public()
cov = gt != "X"                      # 7500 frames com GT
graded = R.graded_mask(N)
yD = (gt == "D").astype(int)
segs = R.gt_dead_segments(gt)        # 4 trechos (início, fim)
weak, weak_src = R.weak_labels(X, I)
rtype = R.restart_type(N)
sp = R.set_pieces()

OUT = os.path.join(HERE, "out", "nb01"); os.makedirs(OUT, exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")
COL = {"H": "#a50044", "A": "#1f77b4", "D": "#111111", "viva": "#4c9be8", "morta": "#d62728"}
minute = lambda fr: np.asarray(fr) / FPS / 60

print("{} frames, {} features, {} avaliados".format(N, X.shape[1], graded.sum()))
print("GT público: frames {}-{}, trechos D:".format(np.flatnonzero(cov)[0], np.flatnonzero(cov)[-1]),
      [(s, e, round((e - s) / FPS, 1)) for s, e in segs])
print("rótulo fraco:", {k: int((weak == k).sum()) for k in (1, 0, -1)}, "| fontes:",
      pd.Series(weak_src[weak >= 0]).value_counts().to_dict())

## H0 — Calibrar o rótulo fraco

O Smart Tagging marca o set piece, mas o `start_frame` corresponde a quê: ao apito, ao reinício ou a nenhum dos dois?
Dois set pieces caem dentro da janela do GT público, e é nesses dois que dá para medir.

In [ ]:
in_gt = sp[(sp.start_frame >= np.flatnonzero(cov)[0]) & (sp.start_frame <= np.flatnonzero(cov)[-1])]
rows = []
for _, e in in_gt.iterrows():
    seg = [(s, t) for s, t in segs if s <= e.start_frame < t]
    if seg:
        s, t = seg[0]
        rows.append(dict(tipo=e.type, start_frame=e.start_frame, D_inicio=s, D_fim=t,
                         apito_ate_start_s=(e.start_frame - s) / FPS,
                         start_ate_reinicio_s=(t - e.start_frame) / FPS,
                         end_frame_menos_reinicio_s=(e.end_frame - t) / FPS))
display(pd.DataFrame(rows))

fig, ax = plt.subplots(figsize=(14, 2.2))
w0, w1 = np.flatnonzero(cov)[[0, -1]]
for lab in "HAD":
    for s, e in R.runs(gt == lab):
        ax.axvspan(minute(s), minute(e), ymin=0.55, ymax=1, color=COL[lab], alpha=0.85, lw=0)
for k, c in ((1, COL["morta"]), (0, COL["viva"])):
    for s, e in R.runs(weak == k):
        if e > w0 and s < w1:
            ax.axvspan(minute(max(s, w0)), minute(min(e, w1)), ymin=0, ymax=0.45, color=c, alpha=0.7, lw=0)
for _, e in in_gt.iterrows():
    ax.axvline(minute(e.start_frame), color="orange", lw=2)
    ax.text(minute(e.start_frame), 1.05, e.type, transform=ax.get_xaxis_transform(), ha="center", fontsize=8)
ax.set_xlim(minute(w0), minute(w1)); ax.set_yticks([0.25, 0.77]); ax.set_yticklabels(["fraco", "GT"])
ax.set_xlabel("minuto do vídeo"); ax.grid(False)
ax.set_title("GT (vermelho = casa, azul = fora, preto = morta) × rótulo fraco (vermelho = morta, azul = viva); laranja = start_frame do set piece")
plt.show()

ct = pd.crosstab(pd.Series(np.where(weak[cov] == 1, "fraco: morta", np.where(weak[cov] == 0, "fraco: viva", "sem rótulo")), name=""),
                 pd.Series(np.where(yD[cov] == 1, "GT: D", "GT: viva"), name=""))
display(ct)
print("precisão do fraco 'morta' no GT: {:.1%} | contaminação do fraco 'viva' por D: {:.1%} | cobertura de D: {:.1%}".format(
    yD[cov][weak[cov] == 1].mean(), yD[cov][weak[cov] == 0].mean(), (weak[cov][yD[cov] == 1] == 1).mean()))

**Leitura.** O `start_frame` cai **dentro** do trecho morto: de 6 a 12 s depois do apito e de 3 a 12 s antes do reinício. O `end_frame`
fica uns 30 s depois e não marca nada. Por isso a janela fraca de bola morta é `[start − 4 s, start + 2 s]`. É estreita e conservadora,
ou seja, precisa mas com cobertura baixa. A fonte "linha" (a bola some a menos de 1 m da linha lateral ou de fundo) completa essa janela.
O rótulo fraco de viva ("a mais de 30 s de qualquer reinício e com o bloco em movimento") tem uns 6% de contaminação, porque há paradas que o
Smart Tagging não marca (lesão, falta sem cobrança tagueada). O trecho D de 20 s em 15889 é um exemplo.

⚠️ **Circularidade.** O rótulo fraco de viva usa `all_speed_mean_3s`, então o AUC "fraco" de qualquer feature de velocidade sai
inflado. Neste notebook, **o juiz é o GT** (com o teste circular) e o rótulo fraco serve só para (a) as médias alinhadas aos 59
set pieces e (b) treinar os eixos supervisionados. O notebook 02 treina com uma regra de viva que não usa os jogadores.

## A visualização mais direta: a linha do tempo

Antes de qualquer redução de dimensão, os 5 minutos do GT com os sinais principais. Os trechos D estão sombreados.

In [ ]:
w0, w1 = np.flatnonzero(cov)[[0, -1]]
fr = np.arange(w0, w1 + 1)
panels = [("all_speed_mean_1s", "velocidade média (1 s) [m/s]"),
          ("polarization_1s", "polarização (1 s)"),
          ("chase_cos_k3_all", "perseguição: cos(v, →bola), 3 mais próx."),
          ("frac_closing", "fração se aproximando da bola"),
          ("ball_centroid_cos_2s", "cos(v bola, v centróide), 2 s"),
          ("teams_centroid_d_m", "distância entre centróides [m]"),
          ("all_length", "comprimento do bloco [m]"),
          ("ball_present", "bola visível no ATD")]
fig, axes = plt.subplots(len(panels) + 1, 1, figsize=(15, 1.55 * (len(panels) + 1)), sharex=True)
for lab in "HAD":
    for s, e in R.runs(gt[fr] == lab):
        axes[0].axvspan(minute(fr[s]), minute(fr[min(e, len(fr) - 1)]), color=COL[lab], lw=0)
axes[0].set_yticks([]); axes[0].set_ylabel("GT", rotation=0, labelpad=15)
for ax, (name, label) in zip(axes[1:], panels):
    y = f(name)[fr]
    if name in ("chase_cos_k3_all", "frac_closing"):
        y = pd.Series(y).rolling(FPS, center=True, min_periods=5).mean().values
    ax.plot(minute(fr), y, lw=0.8, color="k")
    for s, e in segs:
        ax.axvspan(minute(s), minute(e), color=COL["morta"], alpha=0.18, lw=0)
    ax.set_ylabel(label, rotation=0, ha="right", fontsize=8); ax.yaxis.set_label_coords(-0.01, 0.4)
axes[-1].set_xlabel("minuto do vídeo")
fig.suptitle("5 min de GT público — faixas vermelhas = bola morta", y=1.0)
fig.tight_layout(); fig.savefig(os.path.join(OUT, "linha_do_tempo_gt.png"), dpi=120); plt.show()

### Jogo inteiro, interativo

Os 59 set pieces sombreados por tipo e o GT público em preto. Dá zoom; o frame aparece no hover, para abrir o mesmo ponto no `viewer/`.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "notebook_connected"

step = 10
fr = np.flatnonzero(graded)[::step]
fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.02,
                    subplot_titles=["velocidade média 3 s", "polarização 1 s", "perseguição (cos, 3 s)", "bola sumiu na linha [s]"])
sm = lambda a, w: pd.Series(a).rolling(w, center=True, min_periods=1).mean().values
series = [f("all_speed_mean_3s"), f("polarization_1s"), sm(f("chase_cos_k3_all"), 3 * FPS), f("ball_lost_at_line_s")]
for r, y in enumerate(series, 1):
    fig.add_trace(go.Scattergl(x=fr, y=y[fr], mode="lines", line=dict(width=1, color="black"), showlegend=False,
                               hovertemplate="frame %{x}<br>%{y:.2f}"), row=r, col=1)
palette = dict(zip(sorted(sp.type.unique()), ["#e41a1c", "#377eb8", "#4daf4a", "#984ea3", "#ff7f00", "#a65628"]))
for _, e in sp.iterrows():
    fig.add_vrect(x0=e.start_frame - R.PRE_S * FPS, x1=e.start_frame + R.POST_S * FPS, fillcolor=palette[e.type],
                  opacity=0.35, line_width=0, annotation_text=e.type[:6], annotation_position="top left",
                  annotation_font_size=8, row=1, col=1)
for s, e in segs:
    fig.add_vrect(x0=s, x1=e, fillcolor="black", opacity=0.25, line_width=0)
fig.update_layout(height=750, title="Jogo inteiro (1 ponto a cada {} frames) — cores = janelas de set piece; preto = D no GT".format(step),
                  margin=dict(l=40, r=10, t=60, b=30))
fig.update_xaxes(title_text="frame do vídeo", row=4, col=1)
fig.show()

## Ferramenta de teste de hipótese

Para cada feature, `testar(nome, direção)` devolve:

* `auc_gt` e `p_circ`: AUC no GT público e p-valor do deslocamento circular (1000 deslocamentos, de no mínimo 60 s);
* `eventos_gt`: em quantos dos 4 trechos D o Cliff's δ local tem o sinal previsto;
* `eventos_sp`: em quantos dos 59 set pieces do jogo inteiro o sinal é o previsto, mais o Wilcoxon pareado sobre eles;
* um veredito: **aceita** se p_circ < 0,05, a direção bate e ≥ 3/4 dos eventos do GT e ≥ 60% dos set pieces concordam;
  **rejeitada** se a direção é a contrária e significativa; **inconclusiva** no resto.

In [ ]:
from scipy.stats import wilcoxon

alive_ctx = graded & (gt != "D") & (weak != 1)
sp_events = [(int(s - R.PRE_S * FPS), int(s + R.POST_S * FPS)) for s in sp.start_frame]
gt_c, yD_c = cov, yD[cov]
RESULTS = []

def testar(name, direcao, hipotese, n_perm=1000):
    # direcao: '<' = menor na bola morta, '>' = maior
    x = f(name)
    a, p, _ = R.circular_shift_test(yD_c, x[gt_c], n=n_perm)
    sign = -1 if direcao == "<" else 1
    d_gt = R.per_event_delta(x, segs, alive_ctx)
    d_sp = R.per_event_delta(x, sp_events, alive_ctx)
    ok_gt = int(np.sum(np.sign(d_gt) == sign))
    ok_sp = d_sp[~np.isnan(d_sp)]
    frac_sp = np.mean(np.sign(ok_sp) == sign) if len(ok_sp) else np.nan
    try:
        p_sp = wilcoxon(ok_sp, alternative="greater" if sign > 0 else "less").pvalue
    except ValueError:
        p_sp = np.nan
    dir_ok = (a - 0.5) * sign > 0
    if p < 0.05 and dir_ok and ok_gt >= 3 and frac_sp >= 0.6:
        v = "aceita"
    elif p < 0.05 and not dir_ok:
        v = "rejeitada (direção oposta)"
    elif not dir_ok and frac_sp < 0.4:
        v = "rejeitada"
    else:
        v = "inconclusiva"
    row = dict(hipotese=hipotese, feature=name, previsto=direcao, auc_gt=round(a, 3), p_circ=round(p, 4),
               eventos_gt="{}/4".format(ok_gt), delta_gt=np.round(d_gt, 2).tolist(),
               eventos_sp="{:.0%}".format(frac_sp), p_wilcoxon_sp=p_sp, veredito=v)
    RESULTS[:] = [r for r in RESULTS if not (r["feature"] == name and r["hipotese"] == hipotese)] + [row]
    return row

def tabela(h):
    return pd.DataFrame([r for r in RESULTS if r["hipotese"] == h]).drop(columns="hipotese")

# sanidade: deve bater com analise/out/separabilidade.csv (a_speed_mean: 0,863)
print("AUC a_speed_mean (separação) = {:.3f}".format(1 - R.auc(yD_c, f("a_speed_mean")[gt_c])))

## H1 — Desaceleração coletiva: o time "para" quando a bola morre

*Previsão.* A velocidade e a energia cinética caem 1–3 s depois do apito e voltam a subir pouco antes do reinício.

*Teste visual.* Médias alinhadas ao evento, como um PETH (*peri-event time histogram*) da neurociência: cada curva é a feature em torno
da âncora, e a banda é o bootstrap **por evento**. Três âncoras:

1. início de D no GT (4 eventos);
2. fim de D no GT, isto é, o reinício (4 eventos);
3. `start_frame` dos 59 set pieces, separados por tipo.

In [ ]:
def peth(names, before=15, after=15, fname=None):
    starts = [s for s, _ in segs]; ends = [e for _, e in segs]
    sp_by = sp.groupby("type").start_frame.apply(list)
    fig, axes = plt.subplots(len(names), 3, figsize=(15, 2.4 * len(names)), sharex="col")
    axes = np.atleast_2d(axes)
    for r, name in enumerate(names):
        x = f(name)
        for c, (anchors, title) in enumerate([(starts, "início de D (GT, 4)"), (ends, "reinício (fim de D no GT, 4)"), (None, "start_frame do set piece (59)")]):
            ax = axes[r, c]
            if anchors is None:
                for (typ, a), color in zip(sp_by.items(), sns.color_palette("Set1", len(sp_by))):
                    t, M = R.event_aligned(x, a, before, after)
                    ax.plot(t, np.nanmean(M, 0), color=color, lw=1, label="{} ({})".format(typ, len(a)))
                t, M = R.event_aligned(x, list(sp.start_frame), before, after)
                m, lo, hi = R.bootstrap_band(M)
                ax.plot(t, m, color="k", lw=2.2, label="todos"); ax.fill_between(t, lo, hi, color="k", alpha=0.15)
                if r == 0: ax.legend(fontsize=6, ncol=2)
            else:
                t, M = R.event_aligned(x, anchors, before, after)
                for row in M: ax.plot(t, row, color="grey", lw=0.6, alpha=0.7)
                m, lo, hi = R.bootstrap_band(M)
                ax.plot(t, m, color=COL["morta"], lw=2); ax.fill_between(t, lo, hi, color=COL["morta"], alpha=0.15)
            ax.axvline(0, color="k", ls="--", lw=0.8)
            if r == 0: ax.set_title(title, fontsize=10)
            if c == 0: ax.set_ylabel(name, fontsize=8)
    for ax in axes[-1]: ax.set_xlabel("s em relação à âncora")
    fig.tight_layout()
    if fname: fig.savefig(os.path.join(OUT, fname), dpi=110)
    plt.show()

peth(["all_speed_mean_1s", "kinetic_energy", "a_frac_walk", "speed_past5s_max"], fname="peth_h1.png")

In [ ]:
for name, d in [("all_speed_mean", "<"), ("all_speed_mean_3s", "<"), ("a_speed_mean", "<"), ("kinetic_energy", "<"),
                ("all_frac_walk", ">"), ("players_still_s", ">"), ("speed_past5s_max", "<"), ("kinetic_energy_d1s", "<")]:
    testar(name, d, "H1 desaceleração")
tabela("H1 desaceleração")

**Quantificar o "freio".** Para cada um dos 59 set pieces, a velocidade média nos 10 s antes da âncora contra a dos 10 s depois.
O teste é um Wilcoxon pareado: os eventos são independentes, então esse teste é válido.

In [ ]:
x = f("all_speed_mean_1s")
t, M = R.event_aligned(x, list(sp.start_frame), 20, 20)
pre = np.nanmean(M[:, (t >= -20) & (t < -10)], 1); dur = np.nanmean(M[:, (t >= -5) & (t < 2)], 1); post = np.nanmean(M[:, (t >= 5) & (t < 15)], 1)
df = pd.DataFrame({"tipo": sp.type, "antes (−20…−10 s)": pre, "na janela (−5…+2 s)": dur, "depois (+5…+15 s)": post})
print("Wilcoxon antes > janela: p = {:.2g} | janela < depois: p = {:.2g}".format(
    wilcoxon(pre, dur, alternative="greater").pvalue, wilcoxon(dur, post, alternative="less").pvalue))
display(df.groupby("tipo").median().round(2))
fig, ax = plt.subplots(figsize=(6, 4))
for i, row in df.iterrows():
    ax.plot([0, 1, 2], row.values[1:], color=palette.get(row.tipo, "grey") if "palette" in globals() else "grey", alpha=0.5, lw=0.8)
ax.plot([0, 1, 2], df.iloc[:, 1:].median(), color="k", lw=3)
ax.set_xticks([0, 1, 2]); ax.set_xticklabels(df.columns[1:]); ax.set_ylabel("velocidade média [m/s]")
ax.set_title("59 set pieces: o bloco desacelera e acelera de novo"); plt.show()

## H2 — Transição de fase ordem → desordem (Vicsek)

No modelo de Vicsek, a polarização φ é o *parâmetro de ordem*: φ ≈ 1 quando todos correm na mesma direção e φ ≈ 0 quando as direções
são aleatórias. A previsão é que, com a bola viva, o time se move de forma coordenada, e que com a bola morta φ colapsa. Se o jogo tivesse **duas
fases** de verdade, a distribuição de φ seria bimodal, e a susceptibilidade χ = var(φ) teria um pico perto da transição.

*Testes.* AUC e teste circular de φ, ordem nemática, correlação entre vizinhos e χ; BIC de um GMM com 1 e 2 componentes sobre φ; e um
**diagrama de fase**: a densidade em (velocidade, φ), com a bola morta por cima.

In [ ]:
from sklearn.mixture import GaussianMixture
fig, axes = plt.subplots(1, 3, figsize=(16, 4.3))
pol = f("polarization_1s")
ax = axes[0]
sns.kdeplot(pol[cov & (yD == 0)], ax=ax, color=COL["viva"], fill=True, label="GT viva")
sns.kdeplot(pol[cov & (yD == 1)], ax=ax, color=COL["morta"], fill=True, label="GT morta")
sns.kdeplot(pol[graded][::10], ax=ax, color="k", lw=1, ls="--", label="jogo inteiro")
ax.set_xlabel("polarização φ (1 s)"); ax.legend(); ax.set_title("distribuição do parâmetro de ordem")

v = pol[graded][::10]; v = v[~np.isnan(v)].reshape(-1, 1)
bic = {k: GaussianMixture(k, random_state=0).fit(v).bic(v) for k in (1, 2, 3)}
ax.text(0.02, 0.95, "BIC  k=1 {:.0f}\n        k=2 {:.0f}\n        k=3 {:.0f}".format(*bic.values()),
        transform=ax.transAxes, va="top", fontsize=8, family="monospace")

ax = axes[1]
sv = f("all_speed_mean_1s")
sel = graded & (np.arange(N) % 7 == 0)
hb = ax.hexbin(sv[sel], pol[sel], gridsize=45, cmap="Greys", bins="log", mincnt=1)
ax.scatter(sv[cov & (yD == 1)][::3], pol[cov & (yD == 1)][::3], s=4, color=COL["morta"], alpha=0.6, label="D (GT)")
ax.scatter(sv[cov & (yD == 0)][::8], pol[cov & (yD == 0)][::8], s=3, color=COL["viva"], alpha=0.4, label="viva (GT)")
ax.set_xlabel("velocidade média [m/s] (≈ 'temperatura')"); ax.set_ylabel("φ"); ax.legend(markerscale=3)
ax.set_title("diagrama de fase: jogo inteiro (cinza) + GT")

ax = axes[2]
t, M = R.event_aligned(f("polarization_susc_2s"), [s for s, _ in segs] + list(sp.start_frame), 15, 15)
m, lo, hi = R.bootstrap_band(M)
ax.plot(t, m, color="k"); ax.fill_between(t, lo, hi, alpha=0.2, color="k")
t, M = R.event_aligned(f("polarization_susc_2s"), [e for _, e in segs], 15, 15)
ax.plot(t, np.nanmean(M, 0), color=COL["morta"], label="alinhado no reinício (GT)")
ax.axvline(0, ls="--", color="grey"); ax.set_title("susceptibilidade χ = var(φ, 2 s) alinhada"); ax.legend(fontsize=8)
ax.set_xlabel("s em relação à âncora (preto: início D + set pieces)")
fig.tight_layout(); fig.savefig(os.path.join(OUT, "h2_fase.png"), dpi=110); plt.show()

In [ ]:
for name, d in [("all_polarization", "<"), ("polarization_1s", "<"), ("h_polarization", "<"), ("a_polarization", "<"),
                ("nematic_order", "<"), ("neighbor_align_r10", "<"), ("polarization_susc_2s", ">"), ("all_milling", ">")]:
    testar(name, d, "H2 ordem→desordem")
tabela("H2 ordem→desordem")

## H3 — Quimiotaxia: os jogadores param de perseguir a bola

Bactérias sobem um gradiente químico, e os jogadores perto da bola se movem *em direção* a ela. A previsão é que, com a bola morta, o
cosseno entre a velocidade de cada jogador e o vetor até a bola (`chase_cos`) vá para ≈ 0, e que o bloco deixe de "seguir" a bola
(`ball_centroid_cos_2s` ≈ 0). Nos escanteios e faltas os jogadores convergem para a área e não para a bola, então a análise é separada por tipo de reinício.

In [ ]:
names = ["chase_cos_k3_all", "chase_cos_k3_a", "approach_rate_k3", "frac_closing", "ball_centroid_cos_2s", "ball_nearest3_d_m"]
fig, axes = plt.subplots(1, len(names), figsize=(18, 4.2))
cat = np.where(weak == 0, "viva (fraco)", np.where(weak == 1, np.where(rtype != "", rtype, "linha"), ""))
cat_gt = np.where(cov, np.where(yD == 1, "D (GT)", "viva (GT)"), "")
order = ["viva (GT)", "D (GT)", "viva (fraco)", "Throw-in", "Corner Kick", "Goal Kick", "Free Kick", "linha"]
sel = (cat != "") | (cat_gt != "")
for ax, name in zip(axes, names):
    d = pd.concat([pd.DataFrame({"grupo": cat_gt[cov], "v": f(name)[cov]}),
                   pd.DataFrame({"grupo": cat[cat != ""][::3], "v": f(name)[cat != ""][::3]})], ignore_index=True)
    sns.boxplot(data=d, x="grupo", y="v", order=[o for o in order if o in set(d.grupo)], ax=ax, fliersize=0,
                palette=["#4c9be8", "#d62728", "#9ecae1"] + ["#fcae91"] * 5)
    ax.set_title(name, fontsize=9); ax.set_xlabel(""); ax.set_ylabel(""); ax.tick_params(axis="x", rotation=70, labelsize=7)
fig.suptitle("H3 — perseguição da bola: GT (2 primeiras caixas) e rótulo fraco por tipo de reinício")
fig.tight_layout(); fig.savefig(os.path.join(OUT, "h3_perseguicao.png"), dpi=110); plt.show()
peth(["chase_cos_k3_all", "frac_closing", "ball_centroid_cos_2s"], fname="peth_h3.png")

In [ ]:
for name, d in [("chase_cos_k3_all", "<"), ("chase_cos_k3_h", "<"), ("chase_cos_k3_a", "<"), ("approach_rate_k3", "<"),
                ("frac_closing", "<"), ("ball_centroid_cos_2s", "<"), ("ball_nearest3_d_m", ">"), ("chase_past5s_mean", "<")]:
    testar(name, d, "H3 quimiotaxia")
tabela("H3 quimiotaxia")

## H4 — Mobilidade de "microrganismo": balístico → confinado

O deslocamento quadrático médio cresce como MSD(τ) ∝ τ^α: α ≈ 2 é movimento balístico (corrida dirigida), α ≈ 1 é difusivo e α < 1 é confinado.
A previsão é que, com a bola morta, os jogadores ficam "confinados" esperando o reinício e α cai. A magnitude do MSD em 1 s também deve cair.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
lags = np.array([0.2, 1.0, 2.0])
for lab, sel, c in (("viva (GT)", cov & (yD == 0), COL["viva"]), ("D (GT)", cov & (yD == 1), COL["morta"])):
    m = [np.nanmedian(f(n)[sel]) for n in ("msd_0p2s", "msd_1p0s", "msd_2p0s")]
    axes[0].loglog(lags, m, "o-", color=c, label=lab)
axes[0].loglog(lags, 0.3 * lags ** 2, "k:", label="τ² (balístico)"); axes[0].loglog(lags, 1.0 * lags, "k--", lw=0.8, label="τ¹ (difusivo)")
axes[0].set_xlabel("τ [s]"); axes[0].set_ylabel("MSD mediano [m²]"); axes[0].legend(fontsize=8); axes[0].set_title("MSD(τ)")
sns.kdeplot(f("msd_alpha")[cov & (yD == 0)], ax=axes[1], color=COL["viva"], fill=True, label="viva")
sns.kdeplot(f("msd_alpha")[cov & (yD == 1)], ax=axes[1], color=COL["morta"], fill=True, label="D")
axes[1].set_xlabel("α"); axes[1].legend(); axes[1].set_title("expoente α (0,2–2 s)")
sns.kdeplot(np.log10(f("msd_1p0s")[cov & (yD == 0)]), ax=axes[2], color=COL["viva"], fill=True, label="viva")
sns.kdeplot(np.log10(f("msd_1p0s")[cov & (yD == 1)]), ax=axes[2], color=COL["morta"], fill=True, label="D")
axes[2].set_xlabel("log10 MSD(1 s) [m²]"); axes[2].legend(); axes[2].set_title("quanto se deslocam em 1 s")
fig.tight_layout(); fig.savefig(os.path.join(OUT, "h4_msd.png"), dpi=110); plt.show()
for name, d in [("msd_alpha", "<"), ("msd_1p0s", "<"), ("msd_2p0s", "<")]:
    testar(name, d, "H4 confinamento")
tabela("H4 confinamento")

**Leitura esperada.** Nas escalas de 0,2–2 s, α fica perto de 2 tanto com a bola viva quanto com a bola morta: o tracking é suavizado e,
mesmo andando devagar, o jogador anda *em linha reta*. A hipótese "confinado" (α < 1) **não se sustenta** nessas escalas. O que muda é
a *magnitude* do MSD, que é só a velocidade de novo. O resultado negativo vale registrar: isolar α da velocidade exigiria escalas de 5–20 s,
mas aí a janela fica maior que o próprio trecho morto.

## H5 — A forma do bloco muda

Na bola viva os dois times ficam compactos e entrelaçados em torno da bola. Num reinício eles se reorganizam: o bloco se **alonga**
(tiro de meta, goleiro com a bola), os centróides se **afastam** e a largura cai (lateral). Também entram aqui a velocidade radial
(expansão/contração) e o wCentroid.

In [ ]:
peth(["all_length", "teams_centroid_d_m", "all_width", "radial_vel_h"], fname="peth_h5.png")
for name, d in [("all_length", ">"), ("a_length", ">"), ("teams_centroid_d_m", ">"), ("all_width", "<"), ("a_width", "<"),
                ("all_stretch", "<"), ("radial_vel_h", ">"), ("ball_wcentroid_d_m", ">"), ("d_length_1s", ">")]:
    testar(name, d, "H5 forma")
tabela("H5 forma")

## H6 — A bola "some" na linha

Não é hipótese de comportamento coletivo, mas é o candidato óbvio para o primeiro nó da árvore: a bola do ATD é presa dentro do campo
(no máximo 0,9 m fora), então "bola fora" nunca aparece diretamente. O que aparece é a bola **sumir** perto da linha.

In [ ]:
for name, d in [("ball_lost_at_line_s", ">"), ("ball_gap_since_s", ">"), ("ball_last_out_dist_m", ">"), ("ball_present", "<"),
                ("ball_speed_1s", "<"), ("ball_center_d_m", ">")]:
    testar(name, d, "H6 bola na linha")
tabela("H6 bola na linha")

## Redução de dimensão

Três conjuntos de features:

* **jogadores**: as 38 de `explorar_estados.PLAYER_FEATURES`, só movimento, sem bola;
* **jogadores + coletivas**: as 38 mais as novas que não dependem da bola (ordem, MSD, forma, passado);
* **tudo com bola**: as anteriores mais perseguição e bola, só nos frames com bola.

Métodos:

* **não supervisionados**: PCA e t-SNE (e UMAP, se instalado). Respondem se a bola morta forma uma região própria sem ninguém dizer o que ela é;
* **supervisionados**: LDA e PLS-DA, treinados no rótulo **fraco** do jogo inteiro e só *projetados* no GT. Respondem se existe uma direção
  do espaço que separa a bola morta e quanto ela separa no GT, que não foi usado no treino.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.cross_decomposition import PLSRegression
from sklearn.manifold import TSNE

NEW_NOBALL = ["nematic_order", "polarization_1s", "polarization_susc_2s", "neighbor_align_r10", "kinetic_energy",
              "kinetic_energy_d1s", "msd_1p0s", "msd_alpha", "players_still_s", "radial_vel_h", "radial_vel_a",
              "d_stretch_1s", "d_length_1s", "d_width_1s", "speed_past2s_mean", "speed_past5s_max",
              "speed_past5s_trend", "pol_past5s_mean", "pol_past2s_trend", "ke_past5s_mean"]
BALLSET = ["chase_cos_k3_all", "chase_cos_k3_h", "chase_cos_k3_a", "approach_rate_k3", "frac_closing",
           "ball_centroid_cos_2s", "ball_nearest3_d_m", "ball_center_d_m", "ball_touchline_d_m", "ball_goalline_d_m",
           "ball_speed_1s", "ball_wcentroid_d_m"]
SETS = {"jogadores": PLAYER_FEATURES, "jogadores+coletivas": PLAYER_FEATURES + NEW_NOBALL,
        "tudo com bola": PLAYER_FEATURES + NEW_NOBALL + BALLSET}
STEP = 5

def matrix(names, frames):
    M = X[np.ix_(frames, [I[n] for n in names])]
    if names is SETS["jogadores"] or "chase_cos_k3_all" not in names:   # sem bola: imputa a mediana nos raros NaN
        med = np.nanmedian(M, 0); M = np.where(np.isnan(M), med, M)
    return M

frames_s = np.flatnonzero(graded)[::STEP]
frames_s = np.union1d(frames_s, np.flatnonzero(cov)[::2])   # GT com mais densidade
emb = {}
for set_name, names in SETS.items():
    M = matrix(names, frames_s)
    keep = ~np.isnan(M).any(1)
    fr, M = frames_s[keep], M[keep]
    sc = StandardScaler().fit(M); Z = sc.transform(M)
    pca = PCA(random_state=0).fit(Z)
    yw = weak[fr]
    tr = yw >= 0
    lda = LinearDiscriminantAnalysis().fit(Z[tr], yw[tr])
    s_lda = lda.decision_function(Z)
    # 2º eixo: PC1 do resíduo ortogonal ao eixo LDA
    w = lda.scalings_[:, 0] / np.linalg.norm(lda.scalings_[:, 0])
    resid = Z - np.outer(Z @ w, w)
    pc_res = PCA(1, random_state=0).fit(resid[tr])
    pls = PLSRegression(2).fit(Z[tr], yw[tr])
    emb[set_name] = dict(frames=fr, Z=Z, scaler=sc, names=names, pca=pca, P=pca.transform(Z)[:, :2], lda=lda, w=w, pc_res=pc_res,
                         LDA=np.c_[s_lda, pc_res.transform(resid)[:, 0]], PLS=pls.transform(Z), pls=pls)
    g = cov[fr]
    print("{:<22} {:>3} features {:>6} pontos | PCA PC1 {:.0%} PC2 {:.0%} | AUC no GT: LDA {:.3f}, PLS1 {:.3f}, PC1 {:.3f}".format(
        set_name, len(names), len(fr), *pca.explained_variance_ratio_[:2],
        R.auc(yD[fr][g], s_lda[g]), R.auc(yD[fr][g], emb[set_name]["PLS"][g, 0]),
        max(R.auc(yD[fr][g], emb[set_name]["P"][g, 0]), 1 - R.auc(yD[fr][g], emb[set_name]["P"][g, 0]))))

In [ ]:
def tsne_for(set_name, n=9000, seed=0):
    e = emb[set_name]; rng = np.random.default_rng(seed)
    g = np.flatnonzero(cov[e["frames"]])
    sub = np.union1d(rng.choice(len(e["frames"]), min(n, len(e["frames"])), replace=False), g)
    Pk = e["pca"].transform(e["Z"][sub])[:, :int(np.searchsorted(np.cumsum(e["pca"].explained_variance_ratio_), 0.95) + 1)]
    e["tsne_idx"], e["TSNE"] = sub, TSNE(perplexity=40, init="pca", random_state=seed).fit_transform(Pk)
    try:
        import umap
        e["UMAP"] = umap.UMAP(n_neighbors=30, min_dist=0.1, random_state=seed).fit_transform(Pk)
    except ImportError:
        e["UMAP"] = None

for s in SETS: tsne_for(s)
print("UMAP:", "ok" if emb["jogadores"]["UMAP"] is not None else "umap-learn não instalado (pip install umap-learn) — pulado")

In [ ]:
def paint(ax, xy, fr, mode, title):
    if mode == "gt":
        ax.scatter(xy[:, 0], xy[:, 1], s=1, color="#dddddd", rasterized=True)
        g = cov[fr]
        for lab in "HAD":
            s = g & (gt[fr] == lab)
            ax.scatter(xy[s, 0], xy[s, 1], s=4 if lab != "D" else 7, color=COL[lab], label=lab, rasterized=True, alpha=0.8)
    elif mode == "fraco":
        w = weak[fr]
        ax.scatter(xy[w == -1, 0], xy[w == -1, 1], s=1, color="#e5e5e5", rasterized=True)
        ax.scatter(xy[w == 0, 0], xy[w == 0, 1], s=1, color=COL["viva"], alpha=0.35, rasterized=True, label="viva")
        ax.scatter(xy[w == 1, 0], xy[w == 1, 1], s=2, color=COL["morta"], alpha=0.6, rasterized=True, label="morta")
    elif mode == "tipo":
        ax.scatter(xy[:, 0], xy[:, 1], s=1, color="#e5e5e5", rasterized=True)
        for typ, c in palette.items():
            s = rtype[fr] == typ
            ax.scatter(xy[s, 0], xy[s, 1], s=4, color=c, label=typ, rasterized=True)
    elif mode == "velocidade":
        sc = ax.scatter(xy[:, 0], xy[:, 1], s=1, c=f("all_speed_mean_3s")[fr], cmap="viridis", vmax=3.5, rasterized=True)
    ax.set_title(title, fontsize=9); ax.set_xticks([]); ax.set_yticks([])
    if mode in ("gt", "fraco", "tipo"): ax.legend(markerscale=4, fontsize=6, loc="best")

for set_name in SETS:
    e = emb[set_name]
    views = [("PCA", e["P"], e["frames"]), ("LDA × resíduo", e["LDA"], e["frames"]), ("PLS-DA", e["PLS"], e["frames"]),
             ("t-SNE", e["TSNE"], e["frames"][e["tsne_idx"]])]
    if e["UMAP"] is not None: views.append(("UMAP", e["UMAP"], e["frames"][e["tsne_idx"]]))
    modes = ["gt", "fraco", "tipo", "velocidade"]
    fig, axes = plt.subplots(len(views), len(modes), figsize=(4 * len(modes), 3.6 * len(views)))
    for r, (vn, xy, fr) in enumerate(views):
        for c, mode in enumerate(modes):
            paint(axes[r, c], xy, fr, mode, "{} — {}".format(vn, mode))
    fig.suptitle("conjunto: {} ({} features)".format(set_name, len(e["names"])), fontsize=13, y=1.0)
    fig.tight_layout(); fig.savefig(os.path.join(OUT, "proj_{}.png".format(set_name.replace("+", "_").replace(" ", "_"))), dpi=100)
    plt.show()

### O eixo supervisionado no GT

A LDA foi treinada **só** no rótulo fraco. O histograma abaixo mostra a projeção dos frames do GT nesse eixo, e a curva ao lado mostra o
eixo ao longo dos 5 minutos. Se a bola morta "é visível", os trechos D aparecem como platôs.

In [ ]:
e = emb["jogadores+coletivas"]
g = cov[e["frames"]]
fr = e["frames"][g]; s = e["LDA"][g, 0]
fig, axes = plt.subplots(1, 2, figsize=(16, 3.8), gridspec_kw=dict(width_ratios=[1, 3]))
sns.kdeplot(s[yD[fr] == 0], ax=axes[0], fill=True, color=COL["viva"], label="viva")
sns.kdeplot(s[yD[fr] == 1], ax=axes[0], fill=True, color=COL["morta"], label="D")
axes[0].set_xlabel("escore LDA"); axes[0].legend(); axes[0].set_title("AUC no GT = {:.3f}".format(R.auc(yD[fr], s)))
axes[1].plot(minute(fr), pd.Series(s).rolling(5, center=True, min_periods=1).mean(), color="k", lw=0.8)
for a, b in segs: axes[1].axvspan(minute(a), minute(b), color=COL["morta"], alpha=0.2)
axes[1].axhline(0, color="grey", ls=":"); axes[1].set_xlabel("minuto"); axes[1].set_title("escore LDA ao longo do GT (faixas = D)")
plt.tight_layout(); plt.show()

coef = pd.Series(e["w"], index=e["names"]).sort_values(key=np.abs, ascending=False)
print("pesos do eixo LDA (z-score; + = bola morta), top 15:"); display(coef.head(15).round(3).to_frame("peso"))

## Trajetórias: a bola morta é um "laço" no espaço de estados?

PCA e LDA projetam *qualquer* frame, então dá para seguir cada trecho morto frame a frame. Os traços vão de 10 s antes do apito até 10 s
depois do reinício, com cor pelo tempo relativo. O fundo é a densidade do jogo inteiro. A hipótese é que a jogada sai da nuvem viva,
"estaciona" numa região própria e volta no reinício.

In [ ]:
def project(set_name, frames):
    e = emb[set_name]
    M = matrix(e["names"], frames)
    Z = np.nan_to_num(e["scaler"].transform(M))
    resid = Z - np.outer(Z @ e["w"], e["w"])
    return np.c_[e["lda"].decision_function(Z), e["pc_res"].transform(resid)[:, 0]], e["pca"].transform(Z)[:, :2]

set_name = "jogadores+coletivas"
e = emb[set_name]
fig, axes = plt.subplots(2, 4, figsize=(18, 8.5))
for row, key in enumerate(["LDA", "P"]):
    for k, (s0, s1) in enumerate(segs):
        ax = axes[row, k]
        bg = e[key]
        ax.hexbin(bg[:, 0], bg[:, 1], gridsize=50, cmap="Greys", bins="log", mincnt=1, alpha=0.7)
        fr = np.arange(s0 - 10 * FPS, s1 + 10 * FPS, 2)
        lda_xy, pca_xy = project(set_name, fr)
        xy = lda_xy if key == "LDA" else pca_xy
        xy = pd.DataFrame(xy).rolling(5, center=True, min_periods=1).mean().values
        t = (fr - s0) / FPS
        ax.scatter(xy[:, 0], xy[:, 1], c=t, cmap="coolwarm", s=6, zorder=3)
        ax.plot(xy[:, 0], xy[:, 1], color="k", lw=0.4, alpha=0.6, zorder=2)
        i0, i1 = np.searchsorted(fr, s0), np.searchsorted(fr, s1)
        ax.scatter(*xy[i0], marker="X", s=120, color="black", zorder=4, label="apito")
        ax.scatter(*xy[min(i1, len(xy) - 1)], marker="*", s=200, color="gold", edgecolor="k", zorder=4, label="reinício")
        ax.set_title("D {} ({:.0f} s) — {}".format(k + 1, (s1 - s0) / FPS, "LDA × resíduo" if key == "LDA" else "PCA"), fontsize=9)
        if k == 0 and row == 0: ax.legend(fontsize=7)
fig.suptitle("trajetórias dos 4 trechos mortos (azul = antes, vermelho = depois); fundo = jogo inteiro", y=1.0)
fig.tight_layout(); fig.savefig(os.path.join(OUT, "trajetorias_gt.png"), dpi=110); plt.show()

In [ ]:
# trajetória MÉDIA dos 59 set pieces, por tipo, no plano LDA × resíduo
fig, ax = plt.subplots(figsize=(8, 6.5))
ax.hexbin(e["LDA"][:, 0], e["LDA"][:, 1], gridsize=50, cmap="Greys", bins="log", mincnt=1, alpha=0.6)
offs = np.arange(-15 * FPS, 15 * FPS, 5)
for typ, c in palette.items():
    starts = sp.start_frame[sp.type == typ].values
    stack = np.stack([project(set_name, np.clip(s + offs, 0, N - 1))[0] for s in starts])
    m = np.nanmedian(stack, 0)
    m = pd.DataFrame(m).rolling(5, center=True, min_periods=1).mean().values
    ax.plot(m[:, 0], m[:, 1], color=c, lw=2, label="{} ({})".format(typ, len(starts)))
    ax.scatter(*m[0], color=c, marker="o", s=40, edgecolor="k"); ax.scatter(*m[len(offs) // 2], color=c, marker="X", s=90, edgecolor="k")
ax.legend(fontsize=8); ax.set_xlabel("eixo LDA (→ bola morta)"); ax.set_ylabel("PC1 do resíduo")
ax.set_title("trajetória mediana −15 s → +15 s em torno do set piece (● início, ✕ start_frame)")
fig.savefig(os.path.join(OUT, "trajetorias_setpieces.png"), dpi=110); plt.show()

### Explorar no plotly

O plano LDA × resíduo, com hover mostrando frame, minuto, GT, rótulo fraco e tipo de reinício. Um ponto estranho pode ser conferido direto no `viewer/`.

In [ ]:
import plotly.express as px
e = emb["jogadores+coletivas"]
fr = e["frames"]
d = pd.DataFrame({"x": e["LDA"][:, 0], "y": e["LDA"][:, 1], "frame": fr, "minuto": np.round(minute(fr), 2),
                  "rotulo_gt": np.where(cov[fr], gt[fr], "—"), "fraco": np.array(["—", "viva", "morta"])[weak[fr] + 1],
                  "reinicio": np.where(rtype[fr] != "", rtype[fr], "—"), "vel": np.round(f("all_speed_mean_3s")[fr], 2)})
d["cor"] = np.where(d.rotulo_gt != "—", "GT " + d.rotulo_gt, np.where(d.fraco != "—", "fraco " + d.fraco, "sem rótulo"))
d = pd.concat([d[d.cor == "sem rótulo"].sample(frac=0.3, random_state=0), d[d.cor != "sem rótulo"]])
fig = px.scatter(d, x="x", y="y", color="cor", hover_data=["frame", "minuto", "rotulo_gt", "fraco", "reinicio", "vel"], opacity=0.55,
                 color_discrete_map={"GT D": "black", "GT H": "#a50044", "GT A": "#1f77b4", "fraco morta": "#ff7f0e",
                                     "fraco viva": "#9ecae1", "sem rótulo": "#dddddd"}, render_mode="webgl", height=650,
                 labels={"x": "eixo LDA (→ morta)", "y": "PC1 do resíduo"})
fig.update_traces(marker_size=3); fig.show()

## Redundância entre as features fortes

Antes de alimentar a árvore, vale ver o que é a mesma informação com outro nome. Por exemplo, `all_speed_mean`, `kinetic_energy` e `msd_1p0s` medem
a mesma coisa. O dendrograma da correlação de Spearman ajuda a escolher **uma por família**.

In [ ]:
res = pd.DataFrame(RESULTS)
res["sep"] = (res.auc_gt - 0.5).abs()
strong = res.sort_values("sep", ascending=False).drop_duplicates("feature")
strong = strong[strong.sep > 0.15].feature.tolist()
sub = np.flatnonzero(graded)[::25]
corr = pd.DataFrame(X[np.ix_(sub, [I[n] for n in strong])], columns=strong).corr("spearman")
g = sns.clustermap(corr, cmap="RdBu_r", vmin=-1, vmax=1, figsize=(11, 10), method="average", annot=False, xticklabels=True, yticklabels=True)
g.ax_heatmap.tick_params(labelsize=7); plt.show()

# uma por família: gulosa por |AUC-0,5|, descarta |ρ| > 0,8 com uma já escolhida
chosen = []
for n in strong:
    if all(abs(corr.loc[n, c]) <= 0.8 for c in chosen):
        chosen.append(n)
print("representantes (uma por família):", chosen)

## Resumo das hipóteses

In [ ]:
res = pd.DataFrame(RESULTS)
cols = ["hipotese", "feature", "previsto", "auc_gt", "p_circ", "eventos_gt", "eventos_sp", "p_wilcoxon_sp", "veredito"]
show = res[cols].copy(); show["p_wilcoxon_sp"] = show.p_wilcoxon_sp.map(lambda v: "{:.1e}".format(v) if pd.notna(v) else "")
def color(v):
    return "background-color: #c7e9c0" if v == "aceita" else ("background-color: #fcbba1" if v.startswith("rejeitada") else "background-color: #f0f0f0")
display(show.style.map(color, subset=["veredito"]))
res.to_csv(os.path.join(OUT, "hipoteses.csv"), index=False)

verdict = res.groupby("hipotese").veredito.apply(lambda v: "{} aceitas / {} inconclusivas / {} rejeitadas".format(
    (v == "aceita").sum(), (v == "inconclusiva").sum(), v.str.startswith("rejeitada").sum()))
display(verdict.to_frame("features"))

tree_feats = [n for n in chosen if not n.startswith("st_")]
with open(os.path.join(OUT, "features_arvore.json"), "w", encoding="utf-8") as h:
    json.dump({"representantes": tree_feats, "aceitas": res[res.veredito == "aceita"].feature.unique().tolist()}, h, indent=1, ensure_ascii=False)
print("gravado em", OUT)

### Leitura dos resultados (rodada de 2026-11-03)

| hipótese | veredito | evidência |
|---|---|---|
| **H1 desaceleração** | ✅ aceita | velocidade, energia cinética, fração andando: AUC 0,84–0,86 no GT, p_circ = 0,001, 3/4 eventos do GT e 85–88% dos 59 set pieces (Wilcoxon p ≈ 1e-9). O evento que não segue é o tiro de meta (D4), em que os jogadores *correm* para se reposicionar. |
| **H2 ordem → desordem** | ✅ aceita, mas sem transição de fase "dura" | φ e ordem nemática caem em **4/4** eventos do GT e em ~70–75% dos set pieces. A susceptibilidade χ não tem pico (inconclusiva): é um *crossover* gradual, não uma transição abrupta. |
| **H3 quimiotaxia** | ✅ aceita | `chase_cos`, `frac_closing` e `ball_centroid_cos_2s` em 4/4 eventos e 60–77% dos set pieces. O AUC é menor (0,62–0,77), mas a informação é **diferente** da velocidade, porque depende da bola. |
| **H4 confinamento (α < 1)** | ❌ rejeitada | α é *maior* na bola morta (0/4 eventos): os jogadores andam devagar mas em linha reta até a posição do reinício. A magnitude do MSD separa, mas é só velocidade com outro nome. |
| **H5 forma** | ◐ parcial | a distância entre centróides e o wCentroid são aceitos. Comprimento e largura são consistentes no GT (4/4) mas não nos set pieces (~50%), porque **dependem do tipo de reinício**. |
| **H6 bola na linha** | ✅ para a última posição | `ball_last_out_dist_m` e `ball_center_d_m` são aceitas. `ball_lost_at_line_s` é muito precisa quando dispara, mas cobre poucos eventos. |

**A bola morta é visível?**

* **Nas projeções não supervisionadas (PCA, t-SNE), não como ilha.** O PC1 é o eixo "velocidade coletiva", e a bola morta ocupa a
  *borda lenta* da nuvem. No t-SNE os pontos formam "minhocas", que são a continuidade temporal: cada minhoca é uma jogada. As minhocas pretas
  (D) ficam do lado lento, misturadas com posse lenta.
* **No eixo supervisionado (LDA/PLS), sim, parcialmente.** O AUC no GT é ≈ 0,88 só com jogadores e ≈ 0,90 com bola, contra 0,87 do PC1.
  Os pesos da LDA são quase todos de velocidade, ou seja, **as features coletivas novas acrescentam pouco a um classificador linear por frame**.
* **Nas trajetórias, sim, e é a melhor visualização.** O sistema não "salta" para um estado morto. Ele **escorrega** para a ponta lenta
  do eixo durante uns 5–10 s depois do apito, fica ali e sai de forma abrupta logo depois do reinício.

**A consequência mais importante para a árvore (ver o PETH da H1): o fenômeno é assimétrico no tempo.**
O **reinício é abrupto** (a velocidade tem um mínimo exatamente no reinício e acelera em seguida), enquanto o **apito é gradual**
(a velocidade cai ao longo de ~10 s). Um detector por frame que olhe só o passado vai marcar o início do D com atraso. Como a
submissão é offline, a árvore do notebook 02 usa **contexto futuro**: "os próximos 5 s vão ser lentos?" pega o apito, e "o próximo pico de
aceleração" pega o reinício.